# 02. Limpieza, transformación y unión de las fuentes

Aquí se construye el conjunto de datos final del proyecto, en tres etapas: la
limpieza en diez pasos, cada uno registrando cuántas filas descarta y por qué; la
ingeniería de variables, que añade 45 columnas nuevas; y la unión con la fuente 2
por país ISO3 y año.

El criterio que guía todo el cuaderno es que ninguna fila se elimine sin
justificación documentada. El registro completo queda en
`reports/registro_limpieza.csv`, para poder defender cada decisión.

In [1]:
import sys
from pathlib import Path

# Permite importar los modulos del proyecto desde la carpeta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
import numpy as np
from IPython.display import Image, display

import config as cfg
import estilo

estilo.aplicar()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
print("Entorno listo. Raiz del proyecto:", RAIZ.name)

Entorno listo. Raiz del proyecto: Proyecto-final


## El problema de unir las fuentes: los nombres de país

OpenPowerlifting nombra los países en inglés y con convenciones propias (`USA`,
`England`, `Czechia`, `N.Ireland`), mientras que el Banco Mundial y el PNUD usan
códigos ISO 3166-1 alfa-3. Sin traducir, la unión no encuentra nada.

El módulo `paises.py` resuelve el mapeo de unos 200 nombres. Hay dos decisiones que
conviene documentar. Las cuatro naciones británicas (`England`, `Scotland`, `Wales`,
`N.Ireland`) se agregan a `GBR`, porque ni el Banco Mundial ni el PNUD publican
indicadores desagregados por nación constituyente. Y los estados históricos se
asignan a su sucesor principal, de modo que `USSR` pasa a `RUS`.

In [2]:
import paises

for nombre in ["USA", "England", "Czechia", "N.Ireland", "South Korea", "USSR", "Isle of Man"]:
    iso = paises.a_iso3(nombre)
    print(f"  {nombre:<16} -> {iso}  ({paises.a_region(iso)})")

  USA              -> USA  (America del Norte)
  England          -> GBR  (Europa)
  Czechia          -> CZE  (Europa)
  N.Ireland        -> GBR  (Europa)
  South Korea      -> KOR  (Asia)
  USSR             -> RUS  (Europa)
  Isle of Man      -> IMN  (Europa)


In [3]:
# Verificacion: queda algun pais de la fuente 1 sin traducir?
f1 = pd.read_csv(cfg.F1_RAW, low_memory=False, usecols=["MeetCountry", "Country"])
sin_mapear = paises.sin_mapear(f1["MeetCountry"].dropna().unique())
print(f"Paises distintos en la fuente 1: {f1['MeetCountry'].nunique()}")
print(f"Sin traducir a ISO3: {len(sin_mapear)}  {sin_mapear}")

Paises distintos en la fuente 1: 129
Sin traducir a ISO3: 0  []


## La limpieza, paso a paso

Se ejecuta el proceso completo del módulo `transformacion.py`. Cada línea de la
salida muestra cuántas filas entran, cuántas salen y el porcentaje descartado.

El paso 2.3 elimina alrededor del 20% de las filas al quedarse solo con la
modalidad completa, porque comparar un total de SBD con uno de solo press de banca
no tiene sentido analítico.

El paso 2.5 descarta un 7% adicional: son registros sin marca válida en los tres
movimientos, lo que incluye tanto nulos como valores negativos, que en este formato
significan intento fallado.

El paso 2.6 no elimina ninguna fila, y eso es justamente lo interesante. Comprueba
que el total declarado coincide con la suma de los tres levantamientos, con una
tolerancia de 2,5 kg por el redondeo de los discos. Que no descarte nada confirma la
coherencia interna de la fuente.

La edad recibe un trato distinto al resto. Falta en el 38% de los registros, y
eliminar esas filas destruiría gran parte del histórico, así que se conservan con
edad nula y los análisis por edad usan solo el subconjunto con dato. Las edades
imposibles, fuera del rango de 10 a 90 años, se anulan sin eliminar la fila.

In [4]:
import transformacion as tr

bruto = tr.cargar_fuente1()
limpio = tr.limpiar(bruto)


1. CARGA DE LA FUENTE 1


  Registros femeninos en bruto: 1,120,543 x 42 columnas

2. LIMPIEZA


  2.1 Duplicados exactos                 1,120,543 -> 1,119,718  (-825; 0.07%)
  2.2 Fecha invalida                     1,119,718 -> 1,119,718  (-0; 0.00%)


  2.3 Solo modalidad completa (SBD)      1,119,718 ->   894,805  (-224,913; 20.09%)
  2.4 Solo competicion sancionada          894,805 ->   893,094  (-1,711; 0.19%)


  2.5 Marca valida en los 3 movimientos    893,094 ->   828,899  (-64,195; 7.19%)
  2.6 Total coherente con la suma          828,899 ->   828,899  (-0; 0.00%)
  2.7 Peso corporal plausible              828,899 ->   825,298  (-3,601; 0.43%)


  2.8 Total plausible                      825,298 ->   825,297  (-1; 0.00%)
  2.9 Fuerza relativa plausible            825,297 ->   825,296  (-1; 0.00%)
  2.10 Edades imposibles anuladas         553 (fuera de [10, 90] anios)



  RESULTADO DE LA LIMPIEZA: 825,296 filas (73.7% de las originales)


In [5]:
# El registro de trazabilidad, para el informe
registro = pd.DataFrame(tr.registro)
registro[["paso", "filas_antes", "filas_despues", "eliminadas", "pct_eliminado"]]

,paso,filas_antes,filas_despues,eliminadas,pct_eliminado
0,2.1 Duplicados exactos,1120543,1119718,825,0.074
1,2.2 Fecha invalida,1119718,1119718,0,0.000
2,2.3 Solo modalidad completa (SBD),1119718,894805,224913,20.087
3,2.4 Solo competicion sancionada,894805,893094,1711,0.191
4,2.5 Marca valida en los 3 movimientos,893094,828899,64195,7.188
5,2.6 Total coherente con la suma,828899,828899,0,0.000
6,2.7 Peso corporal plausible,828899,825298,3601,0.434
7,2.8 Total plausible,825298,825297,1,0.000
8,2.9 Fuerza relativa plausible,825297,825296,1,0.000
9,2.10 Edad imposible -> nulo,825296,825296,0,0.000


## Ingeniería de variables

De 42 columnas originales se pasa a 87. Los grupos creados son los siguientes.

Temporales: año, mes, trimestre, década y un `id_atleta` anónimo obtenido por hash
del nombre.

Geográficas: código ISO3 y región continental.

De rendimiento: `fuerza_relativa`, que es el total dividido por el peso corporal, y
para cada movimiento sus kilos, su porcentaje del total y su ratio sobre el peso
corporal.

Perfil de fuerza: en qué movimiento destaca cada atleta respecto a la media del
conjunto, tipificando y tomando el máximo.

Ejecución técnica: tasa de acierto sobre los nueve intentos, calculada solo cuando
hay al menos seis registrados.

Trayectoria: número de competición, marca anterior, mejora respecto a la anterior,
mejor marca previa, récord personal y años de trayectoria. Estas variables son
longitudinales, así que exigen ordenar por atleta y fecha.

In [6]:
transformado = tr.transformar(limpio)


3. INGENIERIA DE VARIABLES


  3.1 Tiempo e identificador     : id_atleta, anio, mes, trimestre, decada


  3.2 Geografia                  : iso3, region (0 filas sin ISO3 = 0.000%)
  3.3 Rendimiento                : fuerza_relativa, kg/pct/rel por movimiento, ratios


  3.4 Perfil de fuerza           : perfil_fuerza (arquetipo dominante)
  3.5 Intentos                   : tasa_acierto_intentos (46.7% con detalle completo)


  3.6 Categorias                 : grupo_edad, categoria_peso, tipo_equipamiento, control_antidoping, ambito_federacion
  3.7 Resultado                  : posicion, es_podio, es_primera


  3.8 Trayectoria                : n_competicion, mejora_kg, es_record_personal (227,570 atletas unicas)

  Columnas: 42 -> 87  (+45 nuevas)


In [7]:
# Ejemplo de trayectoria: seguimiento de una atleta con historial largo
veterana = (transformado.groupby("id_atleta").size().idxmax())
cols = ["Date", "n_competicion", "TotalKg", "total_anterior", "mejora_kg",
        "mejor_total_previo", "es_record_personal"]
transformado[transformado["id_atleta"] == veterana][cols].head(12)

,Date,n_competicion,TotalKg,total_anterior,mejora_kg,mejor_total_previo,es_record_personal
419651,1984-03-11,1,267.5,NaN,NaN,NaN,False
419652,1985-03-03,2,317.5,267.5,50.0,267.5,True
419653,1986-03-29,3,350.0,317.5,32.5,317.5,True
419654,1986-05-08,4,355.0,350.0,5.0,350.0,True
419655,1986-12-07,5,355.0,355.0,0.0,355.0,False
419656,1987-03-08,6,370.0,355.0,15.0,355.0,True
419657,1987-05-31,7,375.0,370.0,5.0,370.0,True
419658,1987-12-05,8,385.0,375.0,10.0,375.0,True
419659,1988-02-27,9,395.5,385.0,10.5,385.0,True
419660,1989-02-25,10,382.5,395.5,-13.0,395.5,False


## Unión de las fuentes

Aquí surge un problema real de cobertura. Los indicadores del Banco Mundial y del
PNUD se publican con retraso, de modo que llegan hasta 2022-2024 y arrancan en 1990,
mientras que las competiciones cubren de 1975 a 2026. Uniendo sin más, el 39% de los
registros se quedaba sin contexto socioeconómico.

La decisión tomada es propagar el último valor conocido hacia adelante y el primero
hacia atrás, dentro de cada país. Es defendible porque el IDH, el GII y el PIB per
cápita son series muy inerciales: cambian poco de un año al siguiente. Lo importante
es que cada fila imputada queda marcada en la columna `indicadores_imputados`, de
modo que los análisis que exijan dato observado puedan excluirlas, y así se hace en
la tercera pregunta.

In [8]:
unido = tr.unir_fuentes(transformado)


4. UNION DE LAS DOS FUENTES DE DATOS
  4.1 Banco Mundial (2a)  | +7 columnas | cobertura 99.9% | filas 825,296 -> 825,296


  4.2 PNUD / UNDP   (2b)  | +9 columnas | cobertura 99.7% | filas 825,296


      -> 38.7% de filas con algun indicador propagado (marcado en 'indicadores_imputados')
  4.3 Participacion femenina por pais-anio | cobertura 100.0%


  4.4 Referencia masculina | brecha_total_pct, brecha_fuerza_rel_pct (cobertura 100.0%)
  4.5 Derivadas del cruce  | brecha_idh_genero, grupo_desigualdad, grupo_renta


In [9]:
final = tr.guardar(unido)


5. CONJUNTO DE DATOS FINAL


  FORMA FINAL : 825,296 filas x 83 columnas
  Versionado  : dataset_final_powerlifting_femenino.csv.gz (83.3 MB comprimido)
  Local/PowerBI: dataset_final_powerlifting_femenino.csv (484.9 MB)
  Periodo     : 1975-09-05 a 2026-08-09
  Atletas     : 227,570
  Paises      : 114
  Federaciones: 400

  Requisito >= 50.000 filas : CUMPLE (825,296)
  Requisito >= 20 columnas  : CUMPLE (83)
  Trazabilidad: reports/registro_limpieza.csv


  Diccionario : reports/diccionario_datos.csv


## Verificación de los requisitos

El enunciado exige un conjunto final de al menos 50.000 filas y 20 columnas.

In [10]:
print(f"Filas    : {final.shape[0]:,}   (minimo exigido 50.000  ->  x{final.shape[0]/50_000:.1f})")
print(f"Columnas : {final.shape[1]}        (minimo exigido 20      ->  x{final.shape[1]/20:.1f})")
print(f"Periodo  : {final['fecha'].min():%Y-%m-%d} a {final['fecha'].max():%Y-%m-%d}")
print(f"Atletas  : {final['id_atleta'].nunique():,}")
print(f"Paises   : {final['iso3'].nunique()}")

Filas    : 825,296   (minimo exigido 50.000  ->  x16.5)
Columnas : 83        (minimo exigido 20      ->  x4.2)
Periodo  : 1975-09-05 a 2026-08-09
Atletas  : 227,570
Paises   : 114


In [11]:
# Diccionario de datos: las columnas del conjunto final con su calidad
dic = pd.read_csv(cfg.REPORTS / "diccionario_datos.csv")
print(f"{len(dic)} columnas documentadas")
dic.head(30)

83 columnas documentadas


,columna,tipo,nulos_pct,valores_unicos,ejemplo
0,id_atleta,str,0.00,227570,0dafcd18be77
1,nombre_atleta,string,0.00,227570,Kathy Melcher
2,fecha,datetime64[us],0.00,6866,1975-09-05 00:00:00
3,anio,int32,0.00,51,1975
4,mes,int32,0.00,12,9
5,trimestre,int32,0.00,4,3
6,decada,str,0.00,6,1970s
7,pais_competicion,str,0.00,120,USA
8,iso3,str,0.00,114,USA
9,region,str,0.00,6,America del Norte


El siguiente cuaderno, `03_pregunta1_participacion.ipynb`, aborda la primera
pregunta de análisis.